In [ ]:
pip install google-cloud-bigquery

### Loading both tables (kelmar and sok)

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project="omes-solacc-citizenmatch-01-d")

kelmar_query = """
SELECT *
FROM `omes-solacc-citizenmatch-01-d.citizen_match.kelmar_staging_dataset`
"""

kelmar_df = client.query(kelmar_query).to_dataframe()

print("Kelmar shape:", kelmar_df.shape)

### Inspect SSN format in kelmar

In [ ]:
import pandas as pd
import re

def inspect_ssn(df, ssn_col="SSN"):
    sample = df[df[ssn_col].notna()][ssn_col].head(50)

    results = pd.DataFrame({
        "SSN": sample,
        "length": sample.astype(str).apply(len),
        "has_dash": sample.astype(str).str.contains("-"),
        "has_space": sample.astype(str).str.contains(" "),
        "all_digits": sample.astype(str).str.match(r"^\d+$"),
        "masked": sample.astype(str).str.contains(r"X", case=False)
    })

    return results

inspect_ssn(kelmar_df)

### Checking ssn format in sok without loading full table

In [ ]:
sok_query = """
SELECT SSN
FROM `omes-solacc-citizenmatch-01-d.citizen_match.sok_staging_dataset`
WHERE SSN IS NOT NULL
LIMIT 5
"""

sok_sample_df = client.query(sok_query).to_dataframe()

inspect_ssn(sok_sample_df)

### Encrypt ALL SOK

In [ ]:
import time
import pandas as pd
from google.cloud import bigquery, dlp_v2

# -------------------
# Config
# -------------------
PROJECT_ID = "omes-solacc-citizenmatch-01-d"
LOCATION = "us-central1"
DATASET = "citizen_match"

SOK_SOURCE = f"{PROJECT_ID}.{DATASET}.sok_staging_dataset"

# ✅ Deterministic template (v2)
DLP_TEMPLATE_NAME = "projects/omes-solacc-citizenmatch-01-d/locations/us-central1/deidentifyTemplates/3184577974128662341"

# ✅ NEW destination table (do not overwrite v1)
SOK_TOKEN_TABLE = f"{PROJECT_ID}.{DATASET}.sok_staging_dataset_tokenized_v2"

# Chunk sizes
BQ_CHUNK_ROWS = 100_000
DLP_BATCH_SIZE = 200
MAX_RETRIES = 6

# -------------------
# Clients
# -------------------
bq = bigquery.Client(project=PROJECT_ID)
dlp = dlp_v2.DlpServiceClient()
parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"

# -------------------
# DLP batch tokenization (TABLE item)
# -------------------
def tokenize_ssns_with_dlp_table(ssns, batch_size=DLP_BATCH_SIZE, max_retries=MAX_RETRIES):
    out_tokens = [None] * len(ssns)

    for start in range(0, len(ssns), batch_size):
        end = min(start + batch_size, len(ssns))
        batch = ssns[start:end]

        table_item = {
            "table": {
                "headers": [{"name": "SSN"}],
                "rows": [
                    {"values": [{"string_value": ("" if pd.isna(x) else str(x).strip())}]}
                    for x in batch
                ],
            }
        }

        delay = 1.0
        for _ in range(max_retries):
            try:
                resp = dlp.deidentify_content(
                    request={
                        "parent": parent,
                        "deidentify_template_name": DLP_TEMPLATE_NAME,
                        "item": table_item,
                    }
                )
                tokens = [r.values[0].string_value for r in resp.item.table.rows]
                out_tokens[start:end] = tokens
                break
            except Exception as e:
                msg = str(e)
                if "ResourceExhausted" in msg or "429" in msg:
                    time.sleep(delay)
                    delay *= 2
                    continue
                raise
        else:
            raise RuntimeError(f"Failed after retries for batch {start}:{end}")

    return out_tokens

# -------------------
# Keyset pagination on stable composite key:
# Primary: (Transaction_ID, DLN)
# Tie-breakers to never skip duplicates: (_data_file_date_, SSN)
# NOTE: Transaction_ID can be NULL -> COALESCE to '' so rows are still included.
# -------------------
def fetch_sok_chunk(last_txn=None, last_dln=None, last_file=None, last_ssn=None, limit=BQ_CHUNK_ROWS):
    where = "WHERE SSN IS NOT NULL AND DLN IS NOT NULL"

    if last_txn is not None:
        where += f"""
        AND (
          COALESCE(CAST(Transaction_ID AS STRING), '') > '{last_txn}'
          OR (
            COALESCE(CAST(Transaction_ID AS STRING), '') = '{last_txn}'
            AND (
              CAST(DLN AS STRING) > '{last_dln}'
              OR (
                CAST(DLN AS STRING) = '{last_dln}'
                AND (
                  COALESCE(CAST(_data_file_date_ AS STRING), '') > '{last_file}'
                  OR (
                    COALESCE(CAST(_data_file_date_ AS STRING), '') = '{last_file}'
                    AND CAST(SSN AS STRING) > '{last_ssn}'
                  )
                )
              )
            )
          )
        )
        """

    query = f"""
    SELECT
      COALESCE(CAST(Transaction_ID AS STRING), '') AS Transaction_ID,
      COALESCE(CAST(_data_file_date_ AS STRING), '') AS _data_file_date_,
      Transaction_Date,
      Transaction_Type,

      CAST(DLN AS STRING) AS DLN,
      CAST(SSN AS STRING) AS SSN,

      First_Name, Middle_Name, Last_Name, Suffix,
      Date_of_Birth,

      Residential_Address_Street, Residential_Address_Street_2,
      Residential_Address_Unit_Type, Residential_Address_Unit,
      Residential_Address_City, Residential_Address_State, Residential_Address_Zip,

      Mailing_Address_Street, Mailing_Address_Street_2,
      Mailing_Address_Unit_Type, Mailing_Address_Unit,
      Mailing_Address_City, Mailing_Address_State, Mailing_Address_Zip
    FROM `{SOK_SOURCE}`
    {where}
    ORDER BY
      COALESCE(CAST(Transaction_ID AS STRING), ''),
      CAST(DLN AS STRING),
      COALESCE(CAST(_data_file_date_ AS STRING), ''),
      CAST(SSN AS STRING)
    LIMIT {limit}
    """
    return bq.query(query).to_dataframe(create_bqstorage_client=True)

# -------------------
# Write helper
# -------------------
def write_chunk_to_bq(df, table_id, first_write=False):
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE" if first_write else "WRITE_APPEND"
    )
    job = bq.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()

# -------------------
# Main loop
# -------------------
last_txn = None
last_dln = ""
last_file = ""
last_ssn = ""
first_write = True
total_written = 0

while True:
    chunk = fetch_sok_chunk(
        last_txn=last_txn,
        last_dln=last_dln,
        last_file=last_file,
        last_ssn=last_ssn,
        limit=BQ_CHUNK_ROWS
    )

    if chunk.empty:
        break

    # Tokenize SSN for this chunk
    chunk["SSN"] = chunk["SSN"].astype("string").str.strip()
    chunk["ssn_token"] = tokenize_ssns_with_dlp_table(chunk["SSN"].tolist(), batch_size=DLP_BATCH_SIZE)

    # Compute last4, BUT keep SSN until after cursor update
    chunk["ssn_last4"] = chunk["SSN"].str.replace(r"\D", "", regex=True).str[-4:]

    # Advance cursor using the last row of this chunk (uses the ORDER BY fields)
    last_txn = chunk["Transaction_ID"].iloc[-1]
    last_dln = chunk["DLN"].iloc[-1]
    last_file = chunk["_data_file_date_"].iloc[-1]
    last_ssn = chunk["SSN"].iloc[-1]

    # Drop raw SSN for safety before writing
    chunk = chunk.drop(columns=["SSN"])

    # Write to BigQuery
    write_chunk_to_bq(chunk, SOK_TOKEN_TABLE, first_write=first_write)
    first_write = False

    total_written += len(chunk)

    print(
        f"Wrote {len(chunk):,} rows | Total written: {total_written:,} | "
        f"last_txn={last_txn} last_dln={last_dln} last_file={last_file} last_ssn={last_ssn}"
    )

print(f"✅ Done. Total rows written to {SOK_TOKEN_TABLE}: {total_written:,}")

### Encrypt ALL Kelmar

In [ ]:
import time
import pandas as pd
from google.cloud import bigquery, dlp_v2

# -------------------
# Config
# -------------------
PROJECT_ID = "omes-solacc-citizenmatch-01-d"
LOCATION = "us-central1"
DATASET = "citizen_match"

# ✅ UPDATE THIS to your Kelmar staging table name in BigQuery
KELMAR_SOURCE_TABLE = f"{PROJECT_ID}.{DATASET}.kelmar_staging_dataset"

# Destination table for tokenized Kelmar
KELMAR_TOKEN_TABLE = f"{PROJECT_ID}.{DATASET}.kelmar_staging_dataset_tokenized_v1"

# ✅ Deterministic template (same one used for SOK)
DLP_TEMPLATE_NAME = "projects/omes-solacc-citizenmatch-01-d/locations/us-central1/deidentifyTemplates/3184577974128662341"

DLP_BATCH_SIZE = 200
MAX_RETRIES = 6

# -------------------
# Clients
# -------------------
bq = bigquery.Client(project=PROJECT_ID)
dlp = dlp_v2.DlpServiceClient()
parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"

# -------------------
# DLP batch tokenization (TABLE item)
# -------------------
def tokenize_ssns_with_dlp_table(ssns, batch_size=DLP_BATCH_SIZE, max_retries=MAX_RETRIES):
    out_tokens = [None] * len(ssns)

    for start in range(0, len(ssns), batch_size):
        end = min(start + batch_size, len(ssns))
        batch = ssns[start:end]

        table_item = {
            "table": {
                "headers": [{"name": "SSN"}],
                "rows": [
                    {"values": [{"string_value": ("" if pd.isna(x) else str(x).strip())}]}
                    for x in batch
                ],
            }
        }

        delay = 1.0
        for _ in range(max_retries):
            try:
                resp = dlp.deidentify_content(
                    request={
                        "parent": parent,
                        "deidentify_template_name": DLP_TEMPLATE_NAME,
                        "item": table_item,
                    }
                )
                tokens = [r.values[0].string_value for r in resp.item.table.rows]
                out_tokens[start:end] = tokens
                break
            except Exception as e:
                msg = str(e)
                if "ResourceExhausted" in msg or "429" in msg:
                    time.sleep(delay)
                    delay *= 2
                    continue
                raise
        else:
            raise RuntimeError(f"Failed after retries for batch {start}:{end}")

    return out_tokens

# -------------------
# Read Kelmar (all rows; ~5k)
# -------------------
kelmar_query = f"""
SELECT
  OwnerID,
  PropertyID,
  NameLast,
  NameFirst,
  NameMiddle,
  Address1,
  Address2,
  Address3,
  City,
  State,
  Zip,
  CAST(SSN AS STRING) AS SSN,
  BirthDT,
  CashValue
FROM `{KELMAR_SOURCE_TABLE}`
"""

kelmar_df = bq.query(kelmar_query).to_dataframe(create_bqstorage_client=True)
print("Loaded Kelmar:", kelmar_df.shape)

# -------------------
# Tokenize SSN (deterministic)
# -------------------
kelmar_df["SSN"] = kelmar_df["SSN"].astype("string").str.strip()
kelmar_df["ssn_token"] = tokenize_ssns_with_dlp_table(kelmar_df["SSN"].tolist(), batch_size=DLP_BATCH_SIZE)
kelmar_df["ssn_last4"] = kelmar_df["SSN"].str.replace(r"\D", "", regex=True).str[-4:]

# Drop raw SSN for safety
kelmar_df = kelmar_df.drop(columns=["SSN"])

# -------------------
# Write tokenized Kelmar table (replace)
# -------------------
job = bq.load_table_from_dataframe(
    kelmar_df,
    KELMAR_TOKEN_TABLE,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
job.result()

print(f"✅ Done. Wrote {len(kelmar_df):,} rows to {KELMAR_TOKEN_TABLE}")
kelmar_df.head()

### Cleaning SOK table using our tokenized table

In [ ]:
import re
import unicodedata
import pandas as pd
from google.cloud import bigquery

# -------------------
# CONFIG
# -------------------
PROJECT_ID = "omes-solacc-citizenmatch-01-d"
DATASET = "citizen_match"

SOK_SOURCE = f"{PROJECT_ID}.{DATASET}.sok_staging_dataset_tokenized_v2"
SOK_CLEAN  = f"{PROJECT_ID}.{DATASET}.sok_clean_v1"

bq = bigquery.Client(project=PROJECT_ID)

# -------------------
# CLEANERS (exactly yours)
# -------------------
def rm_accents(x: str) -> str:
    return unicodedata.normalize("NFKD", x).encode("ASCII", "ignore").decode("ASCII")

def collapse_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

PUNCT_NAME_DROP = r"[^A-Z'\-\s]"
PUNCT_ADDR_KEEP = r"[^A-Z0-9'\-\s]"
TITLE_RE = r"\b(MR|MRS|MS|MISS|DR|PROF|REV|SRGT|SGT|OFFICER|ATTY|ATTORNEY|JUDGE)\b"

DIR_MAP = {"NORTH":"N","SOUTH":"S","EAST":"E","WEST":"W","NORTHEAST":"NE","NORTHWEST":"NW","SOUTHEAST":"SE","SOUTHWEST":"SW"}
STREET_MAP = {"STREET":"ST","AVENUE":"AVE","ROAD":"RD","LANE":"LN","DRIVE":"DR","BOULEVARD":"BLVD","COURT":"CT","PLACE":"PL","PARKWAY":"PKWY","CIRCLE":"CIR","HIGHWAY":"HWY"}

def clean_name(val):
    if val is None: return None
    v = str(val).strip()
    if v == "": return None
    v = rm_accents(v).upper()
    v = re.sub(TITLE_RE, "", v)
    v = re.sub(PUNCT_NAME_DROP, " ", v)
    v = collapse_ws(v)
    return v or None

def clean_state(val):
    if val is None: return None
    v = rm_accents(str(val)).upper()
    v = re.sub(r"[^A-Z]", "", v)[:2]
    return v or None

def clean_zip(val):
    if val is None: return None
    digits = re.sub(r"\D", "", str(val))[:5]
    return digits if digits else None

def clean_street(val1, val2=None):
    parts = [val1, val2] if val2 is not None else [val1]
    v = " ".join([str(p).strip() for p in parts if p is not None and str(p).strip() != ""])
    if v.strip() == "": return None
    v = rm_accents(v).upper()
    v = re.sub(r"\bP\.?\s*O\.?\s*B(?:OX)?\b", "PO BOX", v)
    v = re.sub(r"\b(APARTMENT|APT)\b", "APT", v)
    v = re.sub(r"\b(SUITE|STE)\b", "STE", v)
    v = re.sub(r"#\s*([A-Z0-9\-]+)", r"APT \1", v)
    v = re.sub(r"[.,]", "", v)
    v = re.sub(
        r"\b(NORTHWEST|NORTHEAST|SOUTHWEST|SOUTHEAST|NORTH|SOUTH|EAST|WEST)\b",
        lambda m: DIR_MAP[m.group(0)],
        v
    )
    st_pat = r"\b(" + "|".join(map(re.escape, STREET_MAP.keys())) + r")\b"
    v = re.sub(st_pat, lambda m: STREET_MAP[m.group(0)], v)
    v = re.sub(PUNCT_ADDR_KEEP, " ", v)
    v = collapse_ws(v)
    return v or None

# -------------------
# LOAD TOKENIZED SOK
# -------------------
df = bq.query(f"SELECT * FROM `{SOK_SOURCE}`").to_dataframe(create_bqstorage_client=True)
print("Loaded SOK tokenized:", df.shape)

# -------------------
# CLEAN NAMES
# -------------------
df["first_name_clean"]  = df["First_Name"].map(clean_name)
df["middle_name_clean"] = df["Middle_Name"].map(clean_name)
df["last_name_clean"]   = df["Last_Name"].map(clean_name)

df["full_name_clean"] = (
    df["first_name_clean"].fillna("") + " " +
    df["middle_name_clean"].fillna("") + " " +
    df["last_name_clean"].fillna("")
).map(lambda x: collapse_ws(x) if x.strip() != "" else None)

# -------------------
# CLEAN ADDRESS
# -------------------
df["street_clean"] = df.apply(
    lambda r: clean_street(r["Residential_Address_Street"], r["Residential_Address_Street_2"]),
    axis=1
)

df["city_clean"]  = df["Residential_Address_City"].map(clean_name)
df["state_clean"] = df["Residential_Address_State"].map(clean_state)
df["zip_clean"]   = df["Residential_Address_Zip"].map(clean_zip)

# -------------------
# WRITE CLEAN TABLE
# -------------------
job = bq.load_table_from_dataframe(
    df,
    SOK_CLEAN,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
job.result()

print("✅ SOK cleaned table created:", SOK_CLEAN)

### Cleaning Kelmar table using our tokenized table

In [ ]:
import re
import unicodedata
import pandas as pd
from google.cloud import bigquery

# -------------------
# CONFIG
# -------------------
PROJECT_ID = "omes-solacc-citizenmatch-01-d"
DATASET = "citizen_match"

KELMAR_SOURCE = f"{PROJECT_ID}.{DATASET}.kelmar_staging_dataset_tokenized_v1"
KELMAR_CLEAN  = f"{PROJECT_ID}.{DATASET}.kelmar_clean_v1"

bq = bigquery.Client(project=PROJECT_ID)

# -------------------
# CLEANERS (same as SOK)
# -------------------
def rm_accents(x: str) -> str:
    return unicodedata.normalize("NFKD", x).encode("ASCII", "ignore").decode("ASCII")

def collapse_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

PUNCT_NAME_DROP = r"[^A-Z'\-\s]"
PUNCT_ADDR_KEEP = r"[^A-Z0-9'\-\s]"
TITLE_RE = r"\b(MR|MRS|MS|MISS|DR|PROF|REV|SRGT|SGT|OFFICER|ATTY|ATTORNEY|JUDGE)\b"

DIR_MAP = {"NORTH":"N","SOUTH":"S","EAST":"E","WEST":"W","NORTHEAST":"NE","NORTHWEST":"NW","SOUTHEAST":"SE","SOUTHWEST":"SW"}
STREET_MAP = {"STREET":"ST","AVENUE":"AVE","ROAD":"RD","LANE":"LN","DRIVE":"DR","BOULEVARD":"BLVD","COURT":"CT","PLACE":"PL","PARKWAY":"PKWY","CIRCLE":"CIR","HIGHWAY":"HWY"}

def clean_name(val):
    if val is None: return None
    v = str(val).strip()
    if v == "": return None
    v = rm_accents(v).upper()
    v = re.sub(TITLE_RE, "", v)
    v = re.sub(PUNCT_NAME_DROP, " ", v)
    v = collapse_ws(v)
    return v or None

def clean_state(val):
    if val is None: return None
    v = rm_accents(str(val)).upper()
    v = re.sub(r"[^A-Z]", "", v)[:2]
    return v or None

def clean_zip(val):
    if val is None: return None
    digits = re.sub(r"\D", "", str(val))[:5]
    return digits if digits else None

def clean_street(val1, val2=None, val3=None):
    parts = [val1, val2, val3]
    v = " ".join([str(p).strip() for p in parts if p is not None and str(p).strip() != ""])
    if v.strip() == "": return None
    v = rm_accents(v).upper()
    v = re.sub(r"\bP\.?\s*O\.?\s*B(?:OX)?\b", "PO BOX", v)
    v = re.sub(r"\b(APARTMENT|APT)\b", "APT", v)
    v = re.sub(r"\b(SUITE|STE)\b", "STE", v)
    v = re.sub(r"#\s*([A-Z0-9\-]+)", r"APT \1", v)
    v = re.sub(r"[.,]", "", v)
    v = re.sub(
        r"\b(NORTHWEST|NORTHEAST|SOUTHWEST|SOUTHEAST|NORTH|SOUTH|EAST|WEST)\b",
        lambda m: DIR_MAP[m.group(0)],
        v
    )
    st_pat = r"\b(" + "|".join(map(re.escape, STREET_MAP.keys())) + r")\b"
    v = re.sub(st_pat, lambda m: STREET_MAP[m.group(0)], v)
    v = re.sub(PUNCT_ADDR_KEEP, " ", v)
    v = collapse_ws(v)
    return v or None

# -------------------
# LOAD TOKENIZED KELMAR
# -------------------
df = bq.query(f"SELECT * FROM `{KELMAR_SOURCE}`").to_dataframe()
print("Loaded Kelmar tokenized:", df.shape)

# -------------------
# CLEAN NAMES
# -------------------
df["first_name_clean"]  = df["NameFirst"].map(clean_name)
df["middle_name_clean"] = df["NameMiddle"].map(clean_name)
df["last_name_clean"]   = df["NameLast"].map(clean_name)

df["full_name_clean"] = (
    df["first_name_clean"].fillna("") + " " +
    df["middle_name_clean"].fillna("") + " " +
    df["last_name_clean"].fillna("")
).map(lambda x: collapse_ws(x) if x.strip() != "" else None)

# -------------------
# CLEAN ADDRESS
# -------------------
df["street_clean"] = df.apply(
    lambda r: clean_street(r["Address1"], r["Address2"], r["Address3"]),
    axis=1
)

df["city_clean"]  = df["City"].map(clean_name)
df["state_clean"] = df["State"].map(clean_state)
df["zip_clean"]   = df["Zip"].map(clean_zip)

# -------------------
# WRITE CLEAN TABLE
# -------------------
job = bq.load_table_from_dataframe(
    df,
    KELMAR_CLEAN,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
job.result()

print("✅ Kelmar cleaned table created:", KELMAR_CLEAN)

### Deterministic SSN Match (Clean Tables)

In [ ]:
PROJECT_ID = "omes-solacc-citizenmatch-01-d"
DATASET    = "citizen_match"

KELMAR = f"{PROJECT_ID}.{DATASET}.kelmar_clean_v1"
SOK    = f"{PROJECT_ID}.{DATASET}.sok_clean_v1"
OUTPUT = f"{PROJECT_ID}.{DATASET}.deterministic_matches_v1"

bq = bigquery.Client(project=PROJECT_ID)

query = f"""
CREATE OR REPLACE TABLE `{OUTPUT}` AS
SELECT
    -- Kelmar identifiers
    k.OwnerID,
    k.PropertyID,
    k.full_name_clean AS kelmar_name,
    k.BirthDT,
    k.CashValue,

    -- SOK identifiers
    s.DLN,
    s.full_name_clean AS sok_name,
    s._data_file_date_,
    s.Transaction_ID,
    s.Transaction_Date,
    s.Transaction_Type,

    -- shared
    k.ssn_token,

    -- Preliminary bucket; final bucket (DET_AUTO_APPROVE / DET_REVIEW_MINOR / MODERATE / MISMATCH) assigned in combine step
    CASE
        WHEN LOWER(TRIM(k.full_name_clean)) = LOWER(TRIM(s.full_name_clean))
            THEN 'DET_AUTO_APPROVE'
        ELSE 'DET_REVIEW'
    END AS bucket,

    -- Audit flag so reviewers know why it landed in REVIEW
    CASE
        WHEN LOWER(TRIM(k.full_name_clean)) != LOWER(TRIM(s.full_name_clean))
            THEN 'SSN_MATCH_NAME_MISMATCH'
        ELSE NULL
    END AS match_flag

FROM `{KELMAR}` k
JOIN `{SOK}` s
ON k.ssn_token = s.ssn_token
"""

bq.query(query).result()

print("✅ Deterministic match table created:", OUTPUT)

In [ ]:
count_q = f"""
SELECT COUNT(*) AS match_count
FROM `{OUTPUT}`
"""

print(bq.query(count_q).to_dataframe())

### We now isolate:

✅ Matched Kelmar (done)

❌ Unmatched Kelmar → goes to fuzzy stage

❌ SOK SSN-null + unmatched SSN-present

### Identify Unmatched Kelmar

In [ ]:
UNMATCHED = f"{PROJECT_ID}.{DATASET}.kelmar_unmatched_v1"

query = f"""
CREATE OR REPLACE TABLE `{UNMATCHED}` AS
SELECT *
FROM `{PROJECT_ID}.{DATASET}.kelmar_clean_v1`
WHERE ssn_token NOT IN (
    SELECT DISTINCT ssn_token
    FROM `{PROJECT_ID}.{DATASET}.ssn_deterministic_matches_v1`
)
"""

bq.query(query).result()

print("✅ Unmatched Kelmar table created:", UNMATCHED)

In [ ]:
print(
    bq.query(f"SELECT COUNT(*) AS unmatched_count FROM `{UNMATCHED}`")
    .to_dataframe()
)

### Next Step: Build the Correct Fuzzy Candidate Pool

Before we match anything fuzzily, we must define:

What SOK rows are eligible for fuzzy matching?

There are two possible groups:

Group A — SSN NULL rows (376,344)

These never had SSN. Must go to fuzzy.

Group B — SSN-present rows that did NOT match deterministically

These may represent:

SSN typo in Kelmar

SSN mismatch

Token issue

Identity fraud edge cases

We must decide whether to include them in fuzzy stage.

In [ ]:
matched_sok_query = f"""
SELECT COUNT(DISTINCT DLN) AS matched_sok_rows
FROM `{PROJECT_ID}.{DATASET}.ssn_deterministic_matches_v1`
"""
print(bq.query(matched_sok_query).to_dataframe())

### Deterministic SSN Match Summary

Kelmar matched rows: 3,510

Distinct SOK DLNs matched: 2,344

That means:

Some SOK DLNs matched multiple Kelmar properties.

This is expected because:

One citizen can have multiple properties.

Kelmar is property-level.

SOK is person-level (DLN).

So this distribution makes sense.

### Clean SSN NULL SOK Rows

In [ ]:
SOK_NULL_SOURCE = f"{PROJECT_ID}.{DATASET}.sok_staging_dataset"
SOK_NULL_CLEAN  = f"{PROJECT_ID}.{DATASET}.sok_ssn_null_clean_v1"

query = f"""
CREATE OR REPLACE TABLE `{SOK_NULL_CLEAN}` AS
SELECT *
FROM `{SOK_NULL_SOURCE}`
WHERE SSN IS NULL
"""

bq.query(query).result()

print("✅ SSN-null raw subset created:", SOK_NULL_CLEAN)

In [ ]:
import re
import unicodedata
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = "omes-solacc-citizenmatch-01-d"
DATASET = "citizen_match"

SOK_NULL_SOURCE = f"{PROJECT_ID}.{DATASET}.sok_ssn_null_clean_v1"
SOK_NULL_CLEAN  = f"{PROJECT_ID}.{DATASET}.sok_ssn_null_clean_v1"

bq = bigquery.Client(project=PROJECT_ID)

# ---------- cleaners (exact same as before) ----------
def rm_accents(x: str) -> str:
    return unicodedata.normalize("NFKD", x).encode("ASCII", "ignore").decode("ASCII")

def collapse_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

PUNCT_NAME_DROP = r"[^A-Z'\-\s]"
PUNCT_ADDR_KEEP = r"[^A-Z0-9'\-\s]"
TITLE_RE = r"\b(MR|MRS|MS|MISS|DR|PROF|REV|SRGT|SGT|OFFICER|ATTY|ATTORNEY|JUDGE)\b"

DIR_MAP = {"NORTH":"N","SOUTH":"S","EAST":"E","WEST":"W","NORTHEAST":"NE","NORTHWEST":"NW","SOUTHEAST":"SE","SOUTHWEST":"SW"}
STREET_MAP = {"STREET":"ST","AVENUE":"AVE","ROAD":"RD","LANE":"LN","DRIVE":"DR","BOULEVARD":"BLVD","COURT":"CT","PLACE":"PL","PARKWAY":"PKWY","CIRCLE":"CIR","HIGHWAY":"HWY"}

def clean_name(val):
    if val is None: return None
    v = str(val).strip()
    if v == "": return None
    v = rm_accents(v).upper()
    v = re.sub(TITLE_RE, "", v)
    v = re.sub(PUNCT_NAME_DROP, " ", v)
    v = collapse_ws(v)
    return v or None

def clean_state(val):
    if val is None: return None
    v = rm_accents(str(val)).upper()
    v = re.sub(r"[^A-Z]", "", v)[:2]
    return v or None

def clean_zip(val):
    if val is None: return None
    digits = re.sub(r"\D", "", str(val))[:5]
    return digits if digits else None

def clean_street(val1, val2=None):
    parts = [val1, val2]
    v = " ".join([str(p).strip() for p in parts if p is not None and str(p).strip() != ""])
    if v.strip() == "": return None
    v = rm_accents(v).upper()
    v = re.sub(r"\bP\.?\s*O\.?\s*B(?:OX)?\b", "PO BOX", v)
    v = re.sub(r"\b(APARTMENT|APT)\b", "APT", v)
    v = re.sub(r"\b(SUITE|STE)\b", "STE", v)
    v = re.sub(r"#\s*([A-Z0-9\-]+)", r"APT \1", v)
    v = re.sub(r"[.,]", "", v)
    v = re.sub(
        r"\b(NORTHWEST|NORTHEAST|SOUTHWEST|SOUTHEAST|NORTH|SOUTH|EAST|WEST)\b",
        lambda m: DIR_MAP[m.group(0)],
        v
    )
    st_pat = r"\b(" + "|".join(map(re.escape, STREET_MAP.keys())) + r")\b"
    v = re.sub(st_pat, lambda m: STREET_MAP[m.group(0)], v)
    v = re.sub(PUNCT_ADDR_KEEP, " ", v)
    v = collapse_ws(v)
    return v or None

# ---------- load ----------
df = bq.query(f"SELECT * FROM `{SOK_NULL_SOURCE}`").to_dataframe(create_bqstorage_client=True)
print("Loaded SSN-null rows:", df.shape)

# ---------- clean ----------
df["first_name_clean"]  = df["First_Name"].map(clean_name)
df["middle_name_clean"] = df["Middle_Name"].map(clean_name)
df["last_name_clean"]   = df["Last_Name"].map(clean_name)

df["full_name_clean"] = (
    df["first_name_clean"].fillna("") + " " +
    df["middle_name_clean"].fillna("") + " " +
    df["last_name_clean"].fillna("")
).map(lambda x: collapse_ws(x) if x.strip() != "" else None)

df["street_clean"] = df.apply(
    lambda r: clean_street(r["Residential_Address_Street"], r["Residential_Address_Street_2"]),
    axis=1
)

df["city_clean"]  = df["Residential_Address_City"].map(clean_name)
df["state_clean"] = df["Residential_Address_State"].map(clean_state)
df["zip_clean"]   = df["Residential_Address_Zip"].map(clean_zip)

# ---------- write back ----------
job = bq.load_table_from_dataframe(
    df,
    SOK_NULL_CLEAN,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
job.result()

print("✅ SSN-null rows cleaned successfully")

### Final Fuzzy Candidate Pool

In [ ]:
FUZZY_POOL = f"{PROJECT_ID}.{DATASET}.sok_fuzzy_pool_v1"

query = f"""
CREATE OR REPLACE TABLE `{FUZZY_POOL}` AS

-- 1️⃣ SSN NULL cleaned rows
SELECT
    DLN,
    first_name_clean,
    middle_name_clean,
    last_name_clean,
    full_name_clean,
    street_clean,
    city_clean,
    state_clean,
    zip_clean,
    Date_of_Birth,
    NULL AS ssn_token
FROM `{PROJECT_ID}.{DATASET}.sok_ssn_null_clean_v1`

UNION ALL

-- 2️⃣ SSN-present cleaned rows NOT deterministically matched
SELECT
    s.DLN,
    s.first_name_clean,
    s.middle_name_clean,
    s.last_name_clean,
    s.full_name_clean,
    s.street_clean,
    s.city_clean,
    s.state_clean,
    s.zip_clean,
    s.Date_of_Birth,
    s.ssn_token
FROM `{PROJECT_ID}.{DATASET}.sok_clean_v1` s
LEFT JOIN `{PROJECT_ID}.{DATASET}.ssn_deterministic_matches_v1` m
ON s.ssn_token = m.ssn_token
WHERE m.ssn_token IS NULL
"""

bq.query(query).result()

print("✅ Fuzzy candidate pool created:", FUZZY_POOL)

In [ ]:
print(
    bq.query(f"SELECT COUNT(*) AS fuzzy_pool_count FROM `{FUZZY_POOL}`")
    .to_dataframe()
)

### Prepare Kelmar Fuzzy Table (Minimal Schema)

In [ ]:
KELMAR_FUZZY = f"{PROJECT_ID}.{DATASET}.kelmar_fuzzy_v1"

query = f"""
CREATE OR REPLACE TABLE `{KELMAR_FUZZY}` AS
SELECT
    OwnerID,
    PropertyID,
    first_name_clean,
    middle_name_clean,
    last_name_clean,
    full_name_clean,
    street_clean,
    city_clean,
    state_clean,
    zip_clean,
    BirthDT
FROM `{PROJECT_ID}.{DATASET}.kelmar_unmatched_v1`
"""

bq.query(query).result()

print("✅ Kelmar fuzzy table created:", KELMAR_FUZZY)

In [ ]:
print(
    bq.query(f"SELECT COUNT(*) AS kelmar_fuzzy_count FROM `{KELMAR_FUZZY}`")
    .to_dataframe()
)

### Block Strategy 1
Exact ZIP + Exact Last Name + Exact DOB

### Candidate Pair Volume

In [ ]:
BLOCK1 = f"{PROJECT_ID}.{DATASET}.fuzzy_block1_candidates_v1"

query = f"""
CREATE OR REPLACE TABLE `{BLOCK1}` AS
SELECT
    k.OwnerID,
    k.PropertyID,
    k.full_name_clean AS kelmar_name,
    k.street_clean AS kelmar_street,
    k.city_clean AS kelmar_city,
    k.state_clean AS kelmar_state,
    k.zip_clean AS kelmar_zip,
    k.BirthDT,

    s.DLN,
    s.full_name_clean AS sok_name,
    s.street_clean AS sok_street,
    s.city_clean AS sok_city,
    s.state_clean AS sok_state,
    s.zip_clean AS sok_zip,
    s.Date_of_Birth

FROM `{PROJECT_ID}.{DATASET}.kelmar_fuzzy_v1` k
JOIN `{PROJECT_ID}.{DATASET}.sok_fuzzy_pool_v1` s
ON k.zip_clean = s.zip_clean
AND k.last_name_clean = s.last_name_clean
AND k.BirthDT = s.Date_of_Birth
"""

bq.query(query).result()

print("✅ Block 1 candidate table created:", BLOCK1)

In [ ]:
print(
    bq.query(f"SELECT COUNT(*) AS candidate_count FROM `{BLOCK1}`")
    .to_dataframe()
)

### Score Block 1 Candidates

We will compute:

First name similarity

Full name similarity

Street similarity

### Pull Block 1 Candidates into dataframe

In [ ]:
import pandas as pd

BLOCK1 = f"{PROJECT_ID}.{DATASET}.fuzzy_block1_candidates_v1"

block1_df = bq.query(f"SELECT * FROM `{BLOCK1}`").to_dataframe()
print("Loaded Block 1 candidates:", block1_df.shape)
block1_df.head()

### Add Similarity Scoring

In [ ]:
!pip install rapidfuzz

In [ ]:
from rapidfuzz import fuzz

def similarity(a, b):
    if pd.isna(a) or pd.isna(b):
        return 0
    return fuzz.WRatio(a, b)

block1_df["name_score"] = block1_df.apply(
    lambda r: similarity(r["kelmar_name"], r["sok_name"]),
    axis=1
)

block1_df["street_score"] = block1_df.apply(
    lambda r: similarity(r["kelmar_street"], r["sok_street"]),
    axis=1
)

block1_df["composite_score"] = (
    block1_df["name_score"] * 0.6 +
    block1_df["street_score"] * 0.4
)

block1_df.sort_values("composite_score", ascending=False).head(10)

In [ ]:
block1_df.sort_values("composite_score", ascending=False).tail(10)

In [ ]:
block1_df["composite_score"].describe()

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
from google.cloud import bigquery
# Confidence score + bucket (must be before saving to BQ)
block1_df["confidence_score"] = 95 * (block1_df["composite_score"] / 100)
block1_df["bucket"] = np.where(
    block1_df["confidence_score"] >= 90, "FUZZY_AUTO_APPROVE",
    np.where(block1_df["confidence_score"] >= 80, "FUZZY_REVIEW", "FUZZY_REJECT")
)

block1_df["block_name"] = "BLOCK1_ZIP_LAST_DOB"
block1_df["match_stage"] = "FUZZY"
block1_df["scoring_version"] = "v1"
block1_df["created_at"] = datetime.utcnow()

In [ ]:
BLOCK1_RESULTS = f"{PROJECT_ID}.{DATASET}.fuzzy_block1_classified_v1"

block1_df.to_gbq(
    BLOCK1_RESULTS,
    project_id=PROJECT_ID,
    if_exists="replace"
)

print("✅ Block 1 classified results saved:", len(block1_df))

### Block 2: ZIP + Last Name
(no DOB requirement)

In [ ]:
BLOCK2 = f"{PROJECT_ID}.{DATASET}.fuzzy_block2_candidates_v1"

query = f"""
CREATE OR REPLACE TABLE `{BLOCK2}` AS
SELECT
    k.OwnerID,
    k.PropertyID,
    k.full_name_clean AS kelmar_name,
    k.street_clean AS kelmar_street,
    k.city_clean AS kelmar_city,
    k.state_clean AS kelmar_state,
    k.zip_clean AS kelmar_zip,
    k.BirthDT,

    s.DLN,
    s.full_name_clean AS sok_name,
    s.street_clean AS sok_street,
    s.city_clean AS sok_city,
    s.state_clean AS sok_state,
    s.zip_clean AS sok_zip,
    s.Date_of_Birth

FROM `{PROJECT_ID}.{DATASET}.kelmar_fuzzy_v1` k
JOIN `{PROJECT_ID}.{DATASET}.sok_fuzzy_pool_v1` s
ON k.zip_clean = s.zip_clean
AND k.last_name_clean = s.last_name_clean
"""

bq.query(query).result()

print("✅ Block 2 candidate table created:", BLOCK2)

In [ ]:
print(
    bq.query(f"SELECT COUNT(*) AS candidate_count FROM `{BLOCK2}`")
    .to_dataframe()
)

In [ ]:
BLOCK2 = f"{PROJECT_ID}.{DATASET}.fuzzy_block2_candidates_v1"

block2_df = bq.query(f"SELECT * FROM `{BLOCK2}`").to_dataframe()
print("Loaded Block 2 candidates:", block2_df.shape)

In [ ]:
from rapidfuzz import fuzz

def similarity(a, b):
    if pd.isna(a) or pd.isna(b):
        return 0
    return fuzz.WRatio(a, b)

block2_df["name_score"] = block2_df.apply(
    lambda r: similarity(r["kelmar_name"], r["sok_name"]),
    axis=1
)

block2_df["street_score"] = block2_df.apply(
    lambda r: similarity(r["kelmar_street"], r["sok_street"]),
    axis=1
)

block2_df["composite_score"] = (
    block2_df["name_score"] * 0.6 +
    block2_df["street_score"] * 0.4
)

In [ ]:
import numpy as np
from datetime import datetime

# Confidence score first, then bucket based on it
block2_df["confidence_score"] = 85 * (block2_df["composite_score"] / 100)
block2_df["bucket"] = np.where(
    block2_df["confidence_score"] >= 90, "FUZZY_AUTO_APPROVE",
    np.where(block2_df["confidence_score"] >= 80, "FUZZY_REVIEW", "FUZZY_REJECT")
)

block2_df["block_name"] = "BLOCK2_ZIP_LAST"
block2_df["match_stage"] = "FUZZY"
block2_df["scoring_version"] = "v1"
block2_df["created_at"] = datetime.utcnow()

block2_df["bucket"].value_counts()

In [ ]:
BLOCK2_RESULTS = f"{PROJECT_ID}.{DATASET}.fuzzy_block2_classified_v1"

block2_df.to_gbq(
    BLOCK2_RESULTS,
    project_id=PROJECT_ID,
    if_exists="replace"
)

print("✅ Block 2 classified results saved:", len(block2_df))

In [ ]:
auto_block1 = block1_df[block1_df["bucket"] == "FUZZY_AUTO_APPROVE"][["OwnerID","DLN"]]
auto_block2 = block2_df[block2_df["bucket"] == "FUZZY_AUTO_APPROVE"][["OwnerID","DLN"]]

combined_auto = pd.concat([auto_block1, auto_block2]).drop_duplicates()

print("Unique fuzzy auto matches:", combined_auto.shape[0])

### Final Table

In [ ]:
from google.cloud import bigquery
from rapidfuzz import fuzz
import numpy as np
import pandas as pd

# -------------------
# Config
# -------------------
PROJECT_ID = "omes-solacc-citizenmatch-01-d"
DATASET = "citizen_match"

FINAL_TABLE = f"{PROJECT_ID}.{DATASET}.treasury_match_review_v2"

SOK_TOKEN = f"{PROJECT_ID}.{DATASET}.sok_staging_dataset_tokenized_v2"
SOK_CLEAN = f"{PROJECT_ID}.{DATASET}.sok_clean_v1"
KELMAR_CLEAN = f"{PROJECT_ID}.{DATASET}.kelmar_clean_v1"

BLOCK1_TBL = f"{PROJECT_ID}.{DATASET}.fuzzy_block1_classified_v1"
BLOCK2_TBL = f"{PROJECT_ID}.{DATASET}.fuzzy_block2_classified_v1"

DET_TBL = f"{PROJECT_ID}.{DATASET}.ssn_deterministic_matches_v1"

bq = bigquery.Client(project=PROJECT_ID)

# =========================================================
# 0) Build DLN -> Transaction mapping (latest per DLN)
# =========================================================
dln_map = bq.query(f"""
SELECT
  CAST(DLN AS STRING) AS DLN,
  NULLIF(CAST(Transaction_ID AS STRING), '') AS Transaction_ID,
  _data_file_date_
FROM `{SOK_TOKEN}`
WHERE DLN IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY CAST(DLN AS STRING)
  ORDER BY
    _data_file_date_ DESC,
    SAFE_CAST(Transaction_Date AS TIMESTAMP) DESC,
    IF(NULLIF(CAST(Transaction_ID AS STRING), '') IS NULL, 1, 0) ASC
) = 1
""").to_dataframe()

print("DLN map rows:", dln_map.shape)

# =========================================================
# 1) Deterministic matches (join back for full fields + lineage)
# =========================================================
det_full = bq.query(f"""
SELECT
    d.OwnerID,
    d.PropertyID,
    CAST(d.DLN AS STRING) AS DLN,

    k.full_name_clean AS kelmar_name,
    k.street_clean AS kelmar_street,
    k.city_clean AS kelmar_city,
    k.state_clean AS kelmar_state,
    k.zip_clean AS kelmar_zip,
    k.BirthDT,

    sc.full_name_clean AS sok_name,
    sc.street_clean AS sok_street,
    sc.city_clean AS sok_city,
    sc.state_clean AS sok_state,
    sc.zip_clean AS sok_zip,
    sc.Date_of_Birth,

    'DETERMINISTIC_SSN' AS technique,

    CASE
        WHEN LOWER(TRIM(k.full_name_clean)) != LOWER(TRIM(sc.full_name_clean))
            THEN 'SSN_MATCH_NAME_MISMATCH'
        ELSE NULL
    END AS match_flag

FROM `{DET_TBL}` d
JOIN `{KELMAR_CLEAN}` k
  ON d.OwnerID = k.OwnerID AND d.PropertyID = k.PropertyID
JOIN `{SOK_CLEAN}` sc
  ON CAST(d.DLN AS STRING) = CAST(sc.DLN AS STRING)
""").to_dataframe()

det_full["name_score"] = det_full.apply(
    lambda r: fuzz.WRatio(r["kelmar_name"], r["sok_name"])
    if pd.notna(r["kelmar_name"]) and pd.notna(r["sok_name"]) else 0,
    axis=1
)

det_full["street_score"] = det_full.apply(
    lambda r: fuzz.WRatio(r["kelmar_street"], r["sok_street"])
    if pd.notna(r["kelmar_street"]) and pd.notna(r["sok_street"]) else 0,
    axis=1
)

det_full["composite_score"] = (
    det_full["name_score"] * 0.6 +
    det_full["street_score"] * 0.4
)

det_full["confidence_score"] = 100 * (det_full["composite_score"] / 100)

# Assign deterministic buckets
det_full["bucket"] = np.where(
    det_full["match_flag"].isna(), "DET_AUTO_APPROVE",
    np.where(
        det_full["name_score"] >= 90, "DET_REVIEW_MINOR",
        np.where(
            det_full["name_score"] >= 80, "DET_REVIEW_MODERATE",
            "DET_REVIEW_MISMATCH"
        )
    )
)

det_full = det_full.merge(dln_map, on="DLN", how="left")
print("Deterministic rows:", det_full.shape[0])

# =========================================================
# 2) Load fuzzy block tables
# =========================================================
block1 = bq.query(f"SELECT * FROM `{BLOCK1_TBL}`").to_dataframe()
block2 = bq.query(f"SELECT * FROM `{BLOCK2_TBL}`").to_dataframe()

block1["DLN"] = block1["DLN"].astype("string")
block2["DLN"] = block2["DLN"].astype("string")

block1["technique"] = "ZIP+LAST+DOB_STRONG_BLOCK"
block1["confidence_score"] = 95 * (block1["composite_score"] / 100)

block2["technique"] = "ZIP+LAST_FUZZY_BLOCK"
block2["confidence_score"] = 85 * (block2["composite_score"] / 100)

# ✅ NO .map() rename — buckets already correct from classification cells

block1 = block1.merge(dln_map, on="DLN", how="left")
block2 = block2.merge(dln_map, on="DLN", how="left")

# =========================================================
# 3) Standardize column set
# =========================================================
block1["match_flag"] = None
block2["match_flag"] = None

base_cols = [
    "OwnerID","PropertyID","DLN",
    "Transaction_ID","_data_file_date_",
    "kelmar_name","kelmar_street","kelmar_city","kelmar_state","kelmar_zip","BirthDT",
    "sok_name","sok_street","sok_city","sok_state","sok_zip","Date_of_Birth",
    "technique","name_score","street_score","confidence_score","composite_score",
    "bucket","match_flag"
]

for df_name, df in [("det_full", det_full), ("block1", block1), ("block2", block2)]:
    missing = [c for c in base_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} missing columns: {missing}")

det_full = det_full[base_cols]
block1 = block1[base_cols]
block2 = block2[base_cols]

# =========================================================
# 4) Combine + dedupe
# =========================================================
combined = pd.concat([det_full, block1, block2], ignore_index=True)
print("Combined before dedupe:", combined.shape[0])

combined = (
    combined
    .sort_values("confidence_score", ascending=False)
    .drop_duplicates(subset=["OwnerID","PropertyID","DLN"])
)

print("Final unique proposals:", combined.shape[0])

# =========================================================
# ✅ Sanity check — fail loudly if any null buckets
# =========================================================
null_buckets = combined["bucket"].isna().sum()
print(f"\nNull buckets: {null_buckets}")
assert null_buckets == 0, f"❌ Found {null_buckets} null buckets!"

print("\nBucket distribution:")
print(combined["bucket"].value_counts().to_string())

# =========================================================
# 5) Persist to BigQuery
# =========================================================
combined.to_gbq(
    FINAL_TABLE,
    project_id=PROJECT_ID,
    if_exists="replace"
)

print(f"\n✅ {FINAL_TABLE} rebuilt — {combined.shape[0]} rows, 0 null buckets")

### Deterministic always included (high confidence)

Fuzzy limited to 1

No candidate explosion

Still transparent

Treasury can sort by confidence

In [ ]:
from google.cloud import bigquery

PROJECT_ID = "omes-solacc-citizenmatch-01-d"
DATASET = "citizen_match"

SOURCE_TABLE = f"{PROJECT_ID}.{DATASET}.treasury_match_review_v2"
FINAL_CAPPED = f"{PROJECT_ID}.{DATASET}.treasury_match_review_capped_v1"

bq = bigquery.Client(project=PROJECT_ID)

query = f"""
CREATE OR REPLACE TABLE `{FINAL_CAPPED}` AS

WITH ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY OwnerID, PropertyID
      ORDER BY confidence_score DESC
    ) AS rank_within_property
  FROM `{SOURCE_TABLE}`
)

SELECT *
FROM ranked
WHERE
  technique = 'DETERMINISTIC_SSN'
  OR rank_within_property = 1
"""

bq.query(query).result()

print("✅ Final capped Treasury table created:", FINAL_CAPPED)

In [ ]:
bq.query(f"""
SELECT COUNT(*) AS total_rows
FROM `{FINAL_CAPPED}`
""").to_dataframe()

In [ ]:
bq.query(f"""
SELECT COUNT(DISTINCT OwnerID) AS owner_count
FROM `{FINAL_CAPPED}`
""").to_dataframe()

### Build Unmatched Table

In [ ]:
from google.cloud import bigquery

PROJECT_ID = "omes-solacc-citizenmatch-01-d"
DATASET = "citizen_match"

TREASURY_REVIEW = f"{PROJECT_ID}.{DATASET}.treasury_match_review_v2"
KELMAR_CLEAN = f"{PROJECT_ID}.{DATASET}.kelmar_clean_v1"

TREASURY_UNMATCHED = f"{PROJECT_ID}.{DATASET}.treasury_unmatched_v1"

bq = bigquery.Client(project=PROJECT_ID)

query = f"""
CREATE OR REPLACE TABLE `{TREASURY_UNMATCHED}` AS
SELECT
    k.OwnerID,
    k.PropertyID,
    k.full_name_clean AS kelmar_name,
    k.street_clean AS kelmar_street,
    k.city_clean AS kelmar_city,
    k.state_clean AS kelmar_state,
    k.zip_clean AS kelmar_zip,
    k.BirthDT
FROM `{KELMAR_CLEAN}` k
LEFT JOIN `{TREASURY_REVIEW}` r
    ON k.OwnerID = r.OwnerID
WHERE r.OwnerID IS NULL
"""

bq.query(query).result()

print("✅ Treasury unmatched table created:", TREASURY_UNMATCHED)

In [ ]:
bq.query(f"""
SELECT COUNT(*) AS unmatched_count
FROM `{TREASURY_UNMATCHED}`
""").to_dataframe()

### Validate Kelmar values are unchanged
OwnerID exists in original Kelmar table

PropertyID exists

BirthDT matches

Name fields match original cleaned version

In [ ]:
from google.cloud import bigquery

PROJECT_ID = "omes-solacc-citizenmatch-01-d"
DATASET = "citizen_match"

TREASURY_TABLE = f"{PROJECT_ID}.{DATASET}.treasury_match_review_v2"
KELMAR_CLEAN = f"{PROJECT_ID}.{DATASET}.kelmar_clean_v1"

bq = bigquery.Client(project=PROJECT_ID)

query = f"""
SELECT
    COUNT(*) AS total_rows_checked,

    COUNTIF(k.OwnerID IS NULL) AS missing_ownerid,
    COUNTIF(k.PropertyID IS NULL) AS missing_propertyid,

    COUNTIF(t.BirthDT != k.BirthDT) AS birthdt_mismatch,

    COUNTIF(t.kelmar_name != k.full_name_clean) AS name_mismatch,
    COUNTIF(t.kelmar_street != k.street_clean) AS street_mismatch,
    COUNTIF(t.kelmar_city != k.city_clean) AS city_mismatch,
    COUNTIF(t.kelmar_state != k.state_clean) AS state_mismatch,
    COUNTIF(t.kelmar_zip != k.zip_clean) AS zip_mismatch

FROM `{TREASURY_TABLE}` t
LEFT JOIN `{KELMAR_CLEAN}` k
  ON t.OwnerID = k.OwnerID
 AND t.PropertyID = k.PropertyID
"""

result = bq.query(query).to_dataframe()
result

### Validate SOK values are traceable

In [ ]:
from google.cloud import bigquery

PROJECT_ID = "omes-solacc-citizenmatch-01-d"
DATASET = "citizen_match"

TREASURY_TABLE = f"{PROJECT_ID}.{DATASET}.treasury_match_review_v2"
SOK_CLEAN = f"{PROJECT_ID}.{DATASET}.sok_clean_v1"
SOK_NULL_CLEAN = f"{PROJECT_ID}.{DATASET}.sok_ssn_null_clean_v1"

bq = bigquery.Client(project=PROJECT_ID)

# =========================================================
# 1) Overall validation — deduplicated, union of both SOK pools
# =========================================================
overall_query = f"""
WITH sok_all AS (
  -- SSN-present clean rows
  SELECT DLN, full_name_clean, street_clean, city_clean, state_clean, zip_clean,
         Date_of_Birth, _data_file_date_
  FROM `{SOK_CLEAN}`

  UNION ALL

  -- SSN-null clean rows (separate table)
  SELECT CAST(DLN AS STRING) AS DLN, full_name_clean, street_clean, city_clean,
         state_clean, zip_clean, Date_of_Birth,
         CAST(NULL AS STRING) AS _data_file_date_
  FROM `{SOK_NULL_CLEAN}`
),

sok_latest AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY DLN
      ORDER BY _data_file_date_ DESC NULLS LAST
    ) AS rn
  FROM sok_all
)

SELECT
    'ALL' AS technique,
    COUNT(*) AS total_rows,
    COUNTIF(s.DLN IS NULL) AS missing_dln,
    COUNTIF(t.sok_name != s.full_name_clean) AS name_mismatch,
    COUNTIF(t.sok_street != s.street_clean) AS street_mismatch,
    COUNTIF(t.sok_city != s.city_clean) AS city_mismatch,
    COUNTIF(t.sok_state != s.state_clean) AS state_mismatch,
    COUNTIF(t.sok_zip != s.zip_clean) AS zip_mismatch,
    COUNTIF(t.Date_of_Birth != s.Date_of_Birth) AS dob_mismatch
FROM `{TREASURY_TABLE}` t
LEFT JOIN sok_latest s
  ON t.DLN = s.DLN AND s.rn = 1
"""

print("=== OVERALL (latest SOK row per DLN, both pools) ===")
print(bq.query(overall_query).to_dataframe().to_string(index=False))

# =========================================================
# 2) Split by technique — deterministic vs fuzzy
# =========================================================
by_technique_query = f"""
WITH sok_all AS (
  SELECT DLN, full_name_clean, street_clean, city_clean, state_clean, zip_clean,
         Date_of_Birth, _data_file_date_
  FROM `{SOK_CLEAN}`

  UNION ALL

  SELECT CAST(DLN AS STRING) AS DLN, full_name_clean, street_clean, city_clean,
         state_clean, zip_clean, Date_of_Birth,
         CAST(NULL AS STRING) AS _data_file_date_
  FROM `{SOK_NULL_CLEAN}`
),

sok_latest AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY DLN
      ORDER BY _data_file_date_ DESC NULLS LAST
    ) AS rn
  FROM sok_all
)

SELECT
    t.technique,
    COUNT(*) AS total_rows,
    COUNTIF(s.DLN IS NULL) AS missing_dln,
    COUNTIF(t.sok_name != s.full_name_clean) AS name_mismatch,
    COUNTIF(t.sok_street != s.street_clean) AS street_mismatch,
    COUNTIF(t.sok_city != s.city_clean) AS city_mismatch,
    COUNTIF(t.sok_state != s.state_clean) AS state_mismatch,
    COUNTIF(t.sok_zip != s.zip_clean) AS zip_mismatch,
    COUNTIF(t.Date_of_Birth != s.Date_of_Birth) AS dob_mismatch
FROM `{TREASURY_TABLE}` t
LEFT JOIN sok_latest s
  ON t.DLN = s.DLN AND s.rn = 1
GROUP BY t.technique
ORDER BY t.technique
"""

print("\n=== BY TECHNIQUE ===")
print(bq.query(by_technique_query).to_dataframe().to_string(index=False))

# =========================================================
# 3) Fuzzy: check if SOK name+street exist ANYWHERE in history
#    (not just the latest row — addresses change over time)
# =========================================================
fuzzy_history_query = f"""
WITH sok_all AS (
  SELECT DLN, full_name_clean, street_clean
  FROM `{SOK_CLEAN}`

  UNION ALL

  SELECT CAST(DLN AS STRING) AS DLN, full_name_clean, street_clean
  FROM `{SOK_NULL_CLEAN}`
)

SELECT
    COUNT(*) AS fuzzy_rows_checked,

    COUNTIF(name_exists IS FALSE) AS name_not_in_history,
    COUNTIF(street_exists IS FALSE) AS street_not_in_history,
    COUNTIF(name_exists IS TRUE AND street_exists IS TRUE) AS both_confirmed,

    ROUND(COUNTIF(name_exists IS TRUE) * 100.0 / COUNT(*), 1) AS pct_name_confirmed,
    ROUND(COUNTIF(street_exists IS TRUE) * 100.0 / COUNT(*), 1) AS pct_street_confirmed

FROM (
  SELECT
    t.OwnerID,
    t.PropertyID,
    t.DLN,

    EXISTS (
      SELECT 1 FROM sok_all h
      WHERE h.DLN = t.DLN
        AND h.full_name_clean = t.sok_name
    ) AS name_exists,

    EXISTS (
      SELECT 1 FROM sok_all h
      WHERE h.DLN = t.DLN
        AND h.street_clean = t.sok_street
    ) AS street_exists

  FROM `{TREASURY_TABLE}` t
  WHERE t.technique != 'DETERMINISTIC_SSN'
)
"""

print("\n=== FUZZY: name & street exist anywhere in SOK history ===")
print(bq.query(fuzzy_history_query).to_dataframe().to_string(index=False))

### Expanding table columns
CashValue → from kelmar_clean_v1

Deceased → from SOK source tables

In [ ]:
from google.cloud import bigquery

PROJECT_ID = "omes-solacc-citizenmatch-01-d"
DATASET = "citizen_match"

SOURCE = f"{PROJECT_ID}.{DATASET}.treasury_match_review_v2"
FINAL  = f"{PROJECT_ID}.{DATASET}.treasury_match_review_v3"

KELMAR_CLEAN = f"{PROJECT_ID}.{DATASET}.kelmar_clean_v1"
SOK_STAGING  = f"{PROJECT_ID}.{DATASET}.sok_staging_dataset"

bq = bigquery.Client(project=PROJECT_ID)

query = f"""
CREATE OR REPLACE TABLE `{FINAL}` AS

WITH deceased_person AS (
  SELECT
    CAST(DLN AS STRING) AS DLN,
    ANY_VALUE(Deceased) AS Deceased
  FROM `{SOK_STAGING}`
  WHERE DLN IS NOT NULL
  GROUP BY DLN
)

SELECT
    t.*,
    k.CashValue,
    d.Deceased

FROM `{SOURCE}` t

LEFT JOIN `{KELMAR_CLEAN}` k
  ON t.OwnerID = k.OwnerID
 AND t.PropertyID = k.PropertyID

LEFT JOIN deceased_person d
  ON t.DLN = d.DLN
"""

bq.query(query).result()
print("✅ treasury_match_review_v3 rebuilt")

In [ ]:
bq.query(f"""
SELECT COUNT(*) AS total_rows
FROM `{FINAL}`
""").to_dataframe()

In [ ]:
from google.cloud import bigquery

PROJECT_ID = "omes-solacc-citizenmatch-01-d"
DATASET = "citizen_match"

SOURCE = f"{PROJECT_ID}.{DATASET}.treasury_match_review_v3"
FINAL  = f"{PROJECT_ID}.{DATASET}.treasury_match_review_capped_v2"

bq = bigquery.Client(project=PROJECT_ID)

query = f"""
CREATE OR REPLACE TABLE `{FINAL}` AS

WITH enriched AS (
  SELECT
    *,
    CASE 
      WHEN Deceased = TRUE THEN 'INELIGIBLE_DECEASED'
      ELSE 'ELIGIBLE'
    END AS Eligibility_Flag,

    ROW_NUMBER() OVER (
      PARTITION BY OwnerID, PropertyID
      ORDER BY confidence_score DESC
    ) AS property_rank

  FROM `{SOURCE}`
)

SELECT *
FROM enriched
WHERE
  technique = 'DETERMINISTIC_SSN'
  OR property_rank = 1
"""

bq.query(query).result()
print("✅ treasury_match_review_capped_v2 rebuilt")